In [ ]:

import pandas as pd

# 文件路径
file_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv'

# 加载数据
try:
    df = pd.read_csv(file_path)
    print("数据加载成功，前5行如下：")
    print(df.head())
except FileNotFoundError:
    print("文件未找到，请检查文件路径是否正确。")
except Exception as e:
    print(f"数据加载时发生错误：{e}")


数据加载成功，前5行如下：
     id  allelectrons_Total  ...  density_Average  Hardness
0  2124                30.0  ...          0.51006       6.0
1   394                64.0  ...          4.74000       3.3
2  3101                97.0  ...          1.79976       5.3
3  1737               151.0  ...          7.77500       1.8
4   561               131.0  ...          1.92652       5.5

[5 rows x 13 columns]


In [ ]:


# 查看数据集的基本信息
print("数据集基本信息：")
print(df.info())

# 查看目标变量的统计信息
print("\n目标变量 'Hardness' 的统计信息：")
print(df['Hardness'].describe())

# 查看是否有缺失值
print("\n缺失值情况：")
print(df.isnull().sum())


数据集基本信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8325 entries, 0 to 8324
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     8325 non-null   int64  
 1   allelectrons_Total     8325 non-null   float64
 2   density_Total          8325 non-null   float64
 3   allelectrons_Average   8325 non-null   float64
 4   val_e_Average          8325 non-null   float64
 5   atomicweight_Average   8325 non-null   float64
 6   ionenergy_Average      8325 non-null   float64
 7   el_neg_chi_Average     8325 non-null   float64
 8   R_vdw_element_Average  8325 non-null   float64
 9   R_cov_element_Average  8325 non-null   float64
 10  zaratio_Average        8325 non-null   float64
 11  density_Average        8325 non-null   float64
 12  Hardness               8325 non-null   float64
dtypes: float64(12), int64(1)
memory usage: 845.6 KB
None

目标变量 'Hardness' 的统计信息：
count    8325.000000
m

In [ ]:


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 分离特征和目标变量
X = df.drop(columns=['id', 'Hardness'])
y = df['Hardness']

# 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 输出训练集和测试集的形状
print("训练集特征形状：", X_train.shape)
print("测试集特征形状：", X_test.shape)
print("训练集目标形状：", y_train.shape)
print("测试集目标形状：", y_test.shape)



训练集特征形状： (6660, 11)
测试集特征形状： (1665, 11)
训练集目标形状： (6660,)
测试集目标形状： (1665,)


In [ ]:



from sklearn.linear_model import LinearRegression
from sklearn.metrics import median_absolute_error

# 初始化线性回归模型
model = LinearRegression()

# 训练模型
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

# 计算中位绝对误差（Median Absolute Error）
mae = median_absolute_error(y_test, y_pred)

# 输出结果
print(f"线性回归模型在测试集上的中位绝对误差（MAE）：{mae:.4f}")


线性回归模型在测试集上的中位绝对误差（MAE）：0.9259


In [ ]:



from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# 初始化随机森林模型
model = RandomForestRegressor(random_state=42)

# 定义超参数网格
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 使用GridSearchCV进行超参数调优
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_median_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# 获取最佳模型
best_model = grid_search.best_estimator_

# 预测
y_pred = best_model.predict(X_test)

# 计算中位绝对误差（Median Absolute Error）
mae = median_absolute_error(y_test, y_pred)

# 输出结果
print(f"最佳随机森林模型在测试集上的中位绝对误差（MAE）：{mae:.4f}")
print(f"最佳超参数：{grid_search.best_params_}")


最佳随机森林模型在测试集上的中位绝对误差（MAE）：0.5963
最佳超参数：{'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
